In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.proportion import confint_proportions_2indep
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Дизайн A/B-теста (фиксируется до начала проведения эксперимента)

**Контекст:** Команда платежей разработала новую форму оплаты ЖКХ с упрощённым заполнением реквизитов (автозаполнение). Текущая форма требует ручного ввода всех данных. Цель эксперимента — оценить влияние нового интерфейса на конверсию в успешную оплату.

### 1. Гипотезы

- **H₀:** Конверсия из шага 1 в шаг 4 в группе B (новая форма) **равна** конверсии в группе A (текущая форма). Автозаполнение реквизитов не влияет на долю успешных оплат.

- **H₁:** Конверсия из шага 1 в шаг 4 в группе B **выше**, чем в группе A. Новая форма с автозаполнением увеличивает долю успешных оплат.

### 2. Ключевая метрика

**CR (Conversion Rate)** — доля успешных оплат от числа открытий формы:

$$
CR = \frac{\text{step4\_success}}{\text{step1\_opened}}
$$

### 3. Статистические параметры  
* Уровень значимости $\alpha$: 0.05 (ошибка I рода — 5%).
* Мощность ($1 - \beta$): 0.80 ($\beta = 0.20$, ошибка II рода — 20%).
### 4. Распределение пользователей по группам

- **Группа A (контроль):** Пользователи видят текущую форму оплаты (ручной ввод реквизитов).
- **Группа B (тест):** Пользователи видят новую форму с автозаполнением реквизитов.

### 5. Период проведения

- **Дата старта:** 1 апреля 2024 г.
- **Дата окончания:** 30 апреля 2024 г.
- **Длительность:** 30 дней

Период выбран, чтобы собрать достаточно данных и учесть поведение пользователей в разные дни месяца (будни/выходные, начало/конец месяца).  
  
### 6. Метод статистической проверки
Для сравнения пропорций в двух независимых группах используется **двухвыборочный Z-тест для пропорций** (proportions_ztest из библиотеки statsmodels).

**Обоснование метода:**
- Метрика бинарная (успех/неуспех)
- Выборки независимые (пользователи не пересекаются между группами)
- Объём выборок достаточен для применения Z-теста 

### 7. Критерии принятия решения

- Если **p-value < $\alpha$ (0.05)**, отвергаем H₀ и принимаем H₁ — новая форма статистически значимо увеличивает конверсию.
- Если **p-value ≥ $\alpha$ (0.05)**, не отвергаем H₀ — нет достаточных оснований утверждать, что новая форма лучше текущей.

### 8. Дополнительные проверки (сегментный анализ)

Проверим конверсию отдельно по устройствам и городам.  
- **Тип устройства:** iOS / Android
- **Город:** Москва / СПб / Регион

Это позволит убедиться, что наблюдаемый эффект не вызван неравномерным распределением пользователей по этим категориям.


##  Загрузка данных и первичный анализ

In [ ]:
file_path = 'dannye-dlia-keisa-po-pa-cebbefc3-5b29-4f88-b6d4-3336c88d743f.xlsx'
df_users = pd.read_excel(file_path, sheet_name=0)
df_payments = pd.read_excel(file_path, sheet_name=1) 

In [ ]:
df_users.head()

In [ ]:
df_users.info()

 В df_users 1504 записи, есть пропуски в group и city  
 Пропуски в group для нас критинчы, так как мы не можем определить к какой группе относится пользователь, поэтому даннные о таких пользователях удалям.

In [ ]:
df_users = df_users.dropna(subset='group')

Рассмотрим пользователя с пропуском в колонке city:

In [ ]:
df_users[df_users['city'].isnull()]

In [ ]:
df_payments[df_payments['user_id'] == 9004]

 Так как этот пользователь относится к группе А и имеет успешный платеж, что важно для расчета ключевой метрики, данные о нем не стоит удалять для сохранения объема выборки и корректности расчета конверсии. Пропущенное значение в столбце города заменяем на 'Не указан'.

In [ ]:
df_users['city'] = df_users['city'].fillna('Не указан')

In [ ]:
df_users.info()

Проверим данные о пользователях на наличие аномалий:

In [ ]:
df_users['age'].describe()

Можно заметить, что минимальное значение возраста -5, а максимальное 150. Рассмотрим значения выборки более подробно с помощью boxplot и сортировки:

In [ ]:
df_users.boxplot('age')
plt.show()

In [ ]:
df_users['age'].sort_values()

Минимальное значение возраста (-5) и максимальное (150) являются аномалиями.   
Оставшиеся данные распределены в диапазоне 20–60 лет включительно, по ним и произведем фильтрацию:

In [ ]:
df_users = df_users[(df_users['age'] >= 20) & (df_users['age'] <= 60)]

Проверим остальные колонки:

In [ ]:
df_users.device_type.unique()

In [ ]:
df_users.city.unique()

In [ ]:
df_users.group.unique()

Все значения корректны  
Проверим user_id на наличие дубликатов:

In [ ]:
df_users[df_users['user_id'].duplicated(keep=False)]

Дубликатов нет  
Перейдем к первичному анализу данных о платежах

In [ ]:
df_payments.info()

In [ ]:
df_payments = df_payments.dropna(subset='user_id')

In [ ]:
df_payments.info()

Проверим, есть ли операции, которые привязаны к несуществующим user_id (например к тем, которые мы удалили ранее в df_users).

In [ ]:
df_payments[~df_payments['user_id'].isin(df_users['user_id'])]

Проверим, есть ли дублирующиеся значения в payment_id.

In [ ]:
df_payments[df_payments['payment_id'].duplicated(keep=False)]

Проверим, есть ли пользователи, у которых группа в df_payments не совпадает с базовой группой в df_users.

In [ ]:
comparison = df_payments.merge(
    df_users[['user_id', 'group']], 
    on='user_id', 
    how='left',
    suffixes=('_payments', '_users')
)
mismatch = comparison[comparison['group_payments'] != comparison['group_users']]

print(f"Всего платежей с несовпадающей группой: {len(mismatch)}")
if len(mismatch) > 0:
    print(mismatch[['user_id', 'group_payments', 'group_users']].head())

Рассмотрим подробнее пользователя с user_id 1010

In [ ]:
df_users[df_users['user_id'] == 1010]

In [ ]:
df_payments[df_payments['user_id'] == 1010]

У пользователя с user_id 1010 зафиксировано 2 платежа в разных группах (8 апреля в группе B, 27 апреля в группе A). Это указывает на баг сплитования. Удаляем его из обоих датасетов во избежание искажения результатов теста.

In [ ]:
df_users.drop(df_users[df_users['user_id'] == 1010].index, inplace=True)

In [ ]:
df_payments.drop(df_payments[df_payments['user_id'] == 1010].index, inplace=True)

In [ ]:
user_a = df_users[df_users['group'] == 'A']
user_b = df_users[df_users['group'] == 'B']
print(f'Размер группы А: {len(user_a)} пользователей')
print(f'Размер группы B: {len(user_b)} пользователей')

### Итог предобработки:
После очистки аномалий и конфликтов выборка содержит ровно 750 пользователей в группе A и 750 пользователей в группе B.

###  Проверим сбалансированность групп, рассмотрим распределения по городам, по устройствам и по возрасту:

In [ ]:
city_dist = pd.crosstab(df_users['group'], df_users['city'], normalize='index')
print("\nРаспределение по городам:")
print(city_dist.round(3))

device_dist = pd.crosstab(df_users['group'], df_users['device_type'], normalize='index')
print("\nРаспределение по устройствам:")
print(device_dist.round(3))

print("\nСтатистика по возрасту:")
print(df_users.groupby('group')['age'].describe())

В целом группы сбалансированы, но в группе А преобладают пользователи android, а в группе B - ios, поэтому после оценки результатов в общем, рассмотрим еще по сегментам, чтобы убедиться, что нет искажения результатов. (Например, если у пользователей ios форма заполниения работает лучше.

Рассчитаем воронку

In [ ]:
df_funnel = df_payments.groupby('group', as_index=False).agg({
    'step1_opened': 'count',
    'step2_entered': 'count',
    'step3_confirmed': 'count',
    'step4_success': 'count'
})

In [ ]:
df_funnel

In [ ]:
df_funnel['CR'] = df_funnel['step4_success'] / df_funnel['step1_opened']

In [ ]:
df_funnel

In [ ]:
count = df_funnel['step4_success'].tolist()
nobs = df_funnel['step1_opened'].tolist()
z_stat, p_value = proportions_ztest(count, nobs)
print(f"Z-stat: {z_stat:.4f}, p-value: {p_value:.4e}")

In [ ]:
alpha = 0.05

cr_a = df_funnel.loc[df_funnel['group'] == 'A', 'CR'].values[0]
cr_b = df_funnel.loc[df_funnel['group'] == 'B', 'CR'].values[0]

abs_diff = cr_b - cr_a
rel_diff = (cr_b - cr_a) / cr_a

print(f"p-value: {p_value:.4e} | alpha: {alpha}")

if p_value < alpha:
    print("Вывод: Отвергаем H0. Различие в конверсиях статистически значимо.")
else:
    print("Вывод: Принимаем H0. Статистически значимых различий не обнаружено.")

print(f'Конверсия Group A: {cr_a:.2%}')
print(f'Конверсия Group B: {cr_b:.2%}')
print(f'Абсолютный прирост: {abs_diff * 100:+.2f} п.п.')
print(f'Относительный прирост: {rel_diff:+.2%}')

count_b = df_funnel.loc[df_funnel['group'] == 'B', 'step4_success'].values[0]
count_a = df_funnel.loc[df_funnel['group'] == 'A', 'step4_success'].values[0]
nobs_b = df_funnel.loc[df_funnel['group'] == 'B', 'step1_opened'].values[0]
nobs_a = df_funnel.loc[df_funnel['group'] == 'A', 'step1_opened'].values[0]

ci_low, ci_upp = confint_proportions_2indep(
    count1=count_b, nobs1=nobs_b,
    count2=count_a, nobs2=nobs_a,
    method='agresti-caffo', 
    alpha=0.05
)
print(f'При уровне значимости alpha = {alpha} ДИ для разности CR: [{ci_low:.4f}, {ci_upp:.4f}]')

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    data=df_funnel,
    x='group',
    y='CR',
    hue='group',  
    palette={'A': '#4A90D9', 'B': '#FF6B6B'},
    legend=False 
)

plt.title('Сравнение конверсии групп А и B', fontsize=14, fontweight='bold')
plt.xlabel('Группа')
plt.ylabel('Конверсия (CR)')
plt.ylim(0, 0.8)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt='{:.1%}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

Рассмотрим консверсию по всем шагам воронки:

In [ ]:
df_funnel['CR_1_to_2'] = df_funnel['step2_entered'] / df_funnel['step1_opened']
df_funnel['CR_2_to_3'] = df_funnel['step3_confirmed'] / df_funnel['step2_entered']
df_funnel['CR_3_to_4'] = df_funnel['step4_success'] / df_funnel['step3_confirmed']

In [ ]:
df_funnel

In [ ]:
steps = ['Шаг 1→2\nОткрыл форму оплаты→Ввел реквизиты', 'Шаг 2→3\nВвел реквизиты→Подтвердил платеж', 'Шаг 3→4\nПодтвердил платеж→ Успешно оплатил']

cr_a_steps = [df_funnel.loc[0, 'CR_1_to_2'], df_funnel.loc[0, 'CR_2_to_3'], df_funnel.loc[0, 'CR_3_to_4']]
cr_b_steps = [df_funnel.loc[1, 'CR_1_to_2'], df_funnel.loc[1, 'CR_2_to_3'], df_funnel.loc[1, 'CR_3_to_4']]

df_funnel_steps = pd.DataFrame({
    'step': steps * 2,
    'CR': cr_a_steps + cr_b_steps,
    'group': ['A'] * 3 + ['B'] * 3
})

plt.figure(figsize=(12, 6))

sns.barplot(
    data=df_funnel_steps,
    x='step',
    y='CR',
    hue='group',
    palette={'A': '#4A90D9', 'B': '#FF6B6B'}
)

plt.title('Воронка конверсии по шагам (сравнение групп А и Б)', fontsize=14, fontweight='bold')
plt.xlabel('Этап воронки')
plt.ylabel('Конверсия')
plt.ylim(0, 1.0)
plt.legend(title='Группа')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt='{:.1%}', fontsize=11)

plt.tight_layout()
plt.show()

Автозаполнение реквизитов положительно повлияло на всю воронку, увеличив конверсию на каждом этапе.  
Переход к вводу данных вырос на 7,2 п.п., подтверждая, что обновлённый интерфейс стал более простым и понятным.  
Наибольший  прирост показала конверсия перехода от ввода реквизитов к подтверждению платежа — +9.0 п.п.: предзаполненные данные повышают уверенность пользователя при подтверждении оплаты.

Рассмотрим конверсию по сегментам:

In [ ]:
df_payments_full = df_payments.merge(df_users[['user_id', 'age', 'city', 'device_type']], on='user_id', how='left')
funnel_dev = df_payments_full.groupby(['device_type', 'group'], as_index = False).agg(
    s1=('step1_opened', 'count'), s4=('step4_success', 'count'))
funnel_dev['CR'] = funnel_dev['s4'] / funnel_dev['s1']
funnel_dev

In [ ]:
funnel_city = df_payments_full[df_payments_full['city'] != 'Не указан'].groupby(['city', 'group'], as_index=False).agg(
    s1=('step1_opened', 'count'), 
    s4=('step4_success', 'count')
)
funnel_city['CR'] = funnel_city['s4'] / funnel_city['s1']  
funnel_city

In [ ]:
df_payments_full['age_group'] = pd.cut( x= df_payments_full['age'], 
    bins= 4                                    
)
funnel_age  = df_payments_full.groupby(['age_group',  'group'], as_index = False, observed=True).agg(
    s1=('step1_opened', 'count'), 
    s4=('step4_success', 'count') )
funnel_age['CR'] = funnel_age['s4'] / funnel_age['s1']  
funnel_age

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.barplot(
    data=funnel_dev, 
    x='device_type', 
    y='CR', 
    hue='group',
    ax=axes[0],
    palette={'A': '#4A90D9', 'B': '#FF6B6B'}
)
axes[0].set_title('Конверсия по типу устройства', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Тип устройства')
axes[0].set_ylabel('Конверсия (CR)')
axes[0].set_ylim(0, 0.8)
axes[0].legend(title='Группа')

for container in axes[0].containers:
    axes[0].bar_label(container, fmt='{:.1%}', fontsize=10)

sns.barplot(
    data=funnel_city, 
    x='city', 
    y='CR', 
    hue='group',
    ax=axes[1],
    palette={'A': '#4A90D9', 'B': '#FF6B6B'}
)
axes[1].set_title('Конверсия по городам', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Город')
axes[1].set_ylabel('Конверсия (CR)')
axes[1].set_ylim(0, 0.8)
axes[1].legend(title='Группа')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='{:.1%}', fontsize=10)

sns.barplot(
    data=funnel_age, 
    x='age_group', 
    y='CR', 
    hue='group',
    ax=axes[2],
    palette={'A': '#4A90D9', 'B': '#FF6B6B'}
)
axes[2].set_title('Конверсия по возрастным группам', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Возраст')
axes[2].set_ylabel('Конверсия (CR)')
axes[2].set_ylim(0, 0.8)
axes[2].legend(title='Группа')

for container in axes[2].containers:
    axes[2].bar_label(container, fmt='{:.1%}', fontsize=10)

plt.tight_layout()
plt.show()

**Анализ результатов по типу устройств:**  
* Эффект новой формы стабилен для обоих типов устройств. Прирост на Android (+14.9 п.п.) незначительно выше, чем на iOS (+14.1 п.п.), что говорит о том, что автозаполнение реквизитов одинаково эффективно работает на обеих платформах.  
**Анализ результатов по городу:**    
* Наибольший прирост зафиксирован в Москве (+18.2 п.п.), в Санкт-Петербурге прирост также значителен (+14.3 п.п.), в регионах прирост наименьший (+10.7 п.п.), но всё ещё остаётся существенным. Это может объясняться тем, что региональные пользователи, возможно, реже совершают платежи через мобильные приложения, однако положительная динамика подтверждает эффективность новой формы для всех категорий пользователей.
**Анализ результатов по возрастным группам:**  
* Положительный прирост наблюдается во всех возрастных группах, что подтверждает: новый интерфейс удобен и понятен для пользователей разного возраста и не зависит от уровня цифровых навыков. Это говорит о том, что интерфейс новой формы универсален и интуитивно понятен.   
* Для группы 50–60 лет прирост составил всего +4.2 п.п. Однако стоит отметить, что размер выборки в этой возрастной группе крайне мал — всего 24 пользователя в группе A и 12 в группе B. Такие небольшие объёмы не позволяют делать статистически значимые выводы. С учётом малой выборки нельзя утверждать, что эффект для пожилых пользователей слабее — скорее всего, для достоверных выводов требуется больше данных.

Это подтверждает, что наблюдаемый эффект стабилен по всем сегментам не вызван дисбалансом групп.

### Вывод:
Новая форма оплаты с автозаполнением реквизитов показала статистически значимое улучшение ключевой метрики конверсии (CR) на 14.6 п.п. (или +29.3% относительно текущей формы).  

**Оценка эффекта:**   
При $\alpha$ = 0.05 95%-й доверительный интервал разности конверсий составляет [10.85 п.п.; 18.29 п.п.] и находится строго правее нуля. С 95%-й надежностью ожидаемый истинный прирост конверсии оценивается в диапазоне от +10.85 до +18.29 п.п.  
Анализ воронки показал, что улучшение происходит на всех этапах:
* Переход от открытия формы к вводу данных вырос на 7.2 п.п.
* Переход от ввода данных к подтверждению платежа вырос на 9.0 п.п. (наибольший прирост)
* Переход от подтверждения к успешной оплате вырос на 4.7 п.п.

Особенно важно, что наибольший прирост приходится на этап подтверждения платежа: автозаполнение реквизитов повышает уверенность пользователя в правильности введённых данных, что снижает количество отказов на финальном шаге.  

Стабильность эффекта по сегментам: положительная динамика подтверждается во всех анализируемых группах — по типу устройства, городу и возрасту, что свидетельствует об универсальности и устойчивости эффекта новой формы оплаты.

**Рекомендация:**
Запускать новую форму на всех пользователей.

# Дополнительно:
Представим, что у нас была бы другая ключевая метрика, а соответственно другие гипотезы и другой метод статистической проверки (все остальные парамерты теста остаются без изменений). Рассмотрим несколько вариантов тестов с другими метриками:

### Оценка среднего количества успешных попыток на пользователя

### 1. Гипотезы

- **H₀:** Распределение доли успешных попыток на одного пользователя в группе B (новая форма) не отличается от распределения в группе A (текущая форма). Внедренные изменения не влияют на частоту успешных попыток.

- **H₁:** аспределение доли успешных попыток на одного пользователя в группе B отличается от группы A. Внедренные изменения приводят к сдвигу в частоте успешных попыток.

### 2. Ключевая метрика

— отношение суммарного количества успешных попыток к общему числу уникальных пользователей в группе.

### 3. Метод статистической проверки
Поскольку метрика является счётной и чаще всего имеет скошенное распределение (т. к. большинство пользователей совершают 0 или 1 успешную попытку, и лишь немногие — несколько):  
**U-критерий Манна-Уитни** (mannwhitneyu из scipy.stats) — для оценки сдвига распределений без предположения о нормальности.  
В отличие от t-теста, он устойчив к выбросам и оценивает не средние, а сдвиг распределений.

### 4. Критерии принятия решения

- Если **p-value < $\alpha$  (0.05)**, отвергаем H₀ и принимаем H₁ — изменения статистически значимо повлияли на среднее количество успешных попыток.
- Если **p-value ≥ $\alpha$ (0.05)**, не отвергаем H₀ — недостаточно оснований утверждать, что частота успешных попыток изменилась.  
Статистическая значимость по U-критерию говорит о том, что значения в одной группе систематически выше (или ниже), чем в другой. Для интерпретации направления эффекта **необходимо сравнивать медианы групп**.

In [ ]:
df_user_group = df_payments.groupby(['group', 'user_id'], as_index = False).agg(
    all_attempts=('step1_opened', 'nunique'),
    successful_attempts=('step4_success', 'nunique')
)
df_user_group['success_rate'] = df_user_group['successful_attempts'] / df_user_group['all_attempts']
df_user_group.groupby('group', as_index = False).agg({'success_rate' : 'mean' })

In [ ]:
group_a = df_user_group[df_user_group['group'] == 'A']['success_rate']
group_b = df_user_group[df_user_group['group'] == 'B']['success_rate']

res_m = mannwhitneyu(group_b, group_a, alternative='two-sided')

u_stat = res_m.statistic
p_value_m = res_m.pvalue

print(f"""
   Медиана (Группа A): {group_a.median():.4f}
   Медиана (Группа B): {group_b.median():.4f}
   Разность медиан: {group_b.median() - group_a.median():+.4f}
   U-статистика: {u_stat:.4f}
   p-value: {p_value_m:.4e}
""")

### Вывод:  
Средняя доля успешных попыток в группе B (новая форма) составила **65.5%**, что на 15.2 п. п. выше, чем в группе A (текущая форма), где этот показатель равен **50.3%**. Разница в медианах оказалась еще более выраженной: **75% в группе B против 50% в группе A**, что соответствует приросту в **+25 п. п.**. Это означает, что типичный пользователь новой формы успешно завершает 3 из 4 своих попыток оплаты, тогда как в текущей форме — только 1 из 2.  

Результаты теста показали: **U-статистика = 337230.0**, **p-value = 1.8983e-12**. Поскольку p-value значительно меньше уровня значимости $\alpha$ = 0.05, мы **отвергаем нулевую гипотезу H₀** о том, что распределения доли успешных попыток в группах не различаются. Это означает, что различия между группами являются статистически значимыми. Направление эффекта определяется через сравнение медиан: медиана в группе B выше, чем в группе A, следовательно, новая форма приводит к систематическому увеличению доли успешных попыток на пользователя.

Таким образом, новая форма оплаты с автозаполнением реквизитов **статистически значимо повышает долю успешных попыток на пользователя**  
Этот результат согласуется с основным выводом по конверсии и подтверждает, что упрощение ввода данных помогает пользователям успешнее завершать платежи, снижая количество неудачных или прерванных попыток.

##  Оценка среднего времени от момента открытия формы до завершения успешного платежа

### 1. Гипотезы

- **H₀:** Среднее время прохождения формы оплаты (от открытия step1_opened до успешной оплаты step4_success) в группе B (новая форма) **равно** среднему времени в группе A (текущая форма). Автозаполнение реквизитов не влияет на скорость оплаты.

- **H₁:** Среднее время прохождения формы оплаты в группе B **ниже**, чем в группе A. Новая форма с автозаполнением ускоряет процесс оплаты.

### 2. Ключевая метрика

— время от момента открытия формы до завершения успешного платежа (в секундах).

### 3. Метод статистической проверки
Для сравнения средних значений непрерывной метрики в двух независимых группах используется **двухвыборочный t-критерий Стьюдента** (ttest_ind из библиотеки scipy.stats).

**Обоснование метода:**
- Выборки независимые.
- Позволяет оценить статистическую значимость разности средних значений времени прохождения воронки.
- Используется модификация t-критерия — тест Уэлча (параметр equal_var=False). Это обусловлено тем, что дисперсии времени в группах A и B могут различаться (например, новая форма с автозаполнением может не только уменьшить среднее время, но и изменить его вариативность). Тест Уэлча не предполагает равенства дисперсий и дает более надежные результаты в таких условиях, особенно при неравных объемах выборок или различных разбросах данных.

### 4. Критерии принятия решения

- Если **p-value < $\alpha$ (0.05)** и среднее время в группе B меньше, чем в группе A, отвергаем H₀ и принимаем H₁ — новая форма статистически значимо ускоряет процесс оплаты.
- Если **p-value ≥ $\alpha$ (0.05)**, не отвергаем H₀ — нет достаточных оснований утверждать, что новая форма уменьшает время оплаты.


In [ ]:
df_success = df_payments[df_payments['step4_success'].notnull()].copy()
df_success['duration_sec'] = (df_success['step4_success'] - df_success['step1_opened']).dt.total_seconds()

time_A = df_success[df_success['group'] == 'A']['duration_sec']
time_B = df_success[df_success['group'] == 'B']['duration_sec']

t_stat, p_val = ttest_ind(time_A, time_B, equal_var=False, alternative='less')

print(f"Среднее время А: {time_A.mean():.2f} сек")
print(f"Среднее время B: {time_B.mean():.2f} сек")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.5f}")

In [ ]:
se_diff = np.sqrt((time_A.var() / len(time_A)) + (time_B.var() / len(time_B)))

mean_diff = time_B.mean() - time_A.mean()

df = (time_A.var()/len(time_A) + time_B.var()/len(time_B))**2 / (
    (time_A.var()/len(time_A))**2 / (len(time_A)-1) + 
    (time_B.var()/len(time_B))**2 / (len(time_B)-1)
)
t_crit_one_sided = stats.t.ppf(0.95, df)

ci_upper = mean_diff + t_crit_one_sided * se_diff

print(f"Разность средних (B - A): {mean_diff:.2f} сек")
print(f"Односторонний 95% ДИ (B - A): (-∞; {ci_upper:.2f} сек]")

### Вывод 

Среднее время в группе B (новая форма) составило 741.24 секунды, что на 3.92 секунды меньше, чем в группе A (текущая форма), где среднее время равно 745.16 секунд. Разница составляет менее 4 секунд (~0.5% от общего времени).

**Результаты теста**:
* t-статистика: 0.4542
* p-value (односторонний, B < A): 0.67512

Поскольку p-value (0.67512) значительно превышает уровень значимости α = 0.05, у нас нет оснований отвергнуть нулевую гипотезу H₀ о том, что среднее время прохождения формы в группах A и B не различается.

95% односторонний доверительный интервал для разности средних (B − A) составляет (-∞; +10.29 сек]. Поскольку верхняя граница интервала положительна, мы не можем утверждать, что новая форма действительно ускоряет процесс — истинная разница может быть как отрицательной (ускорение), так и положительной (замедление).

Таким образом, измеримого эффекта на время успешного прохождения платежа не обнаружено. Основная ценность новой формы заключается в повышении конверсии и снижении числа ошибок, а не в ускорении. Пользователи, вероятно, тратят сэкономленное на вводе время на более внимательную проверку автозаполненных данных, что нивелирует потенциальный выигрыш в скорости.